# 05 - Synthetic Transaction Stream Generation

This notebook implements synthetic transaction generation to simulate realistic cardholder spending and inject distinct fraud attack scenarios.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
from src.producer.transaction_generator import TransactionGenerator
from config.config import SYNTHETIC_DATA_PATH

gen = TransactionGenerator()
print('Generating normal baseline transactions:')
normal_txns = [gen.generate_normal_transaction() for _ in range(5)]
print(json.dumps(normal_txns[:2], indent=2))

Generating normal baseline transactions:
[
  {
    "transaction_id": "TX10001",
    "card_id": "CARD_021",
    "amount": 81.12,
    "merchant": "Starbucks",
    "location": "Seattle",
    "timestamp": "2026-09-28T10:01:47.373035",
    "status": "SUCCESS",
    "scenario": "NORMAL_SPEND"
  },
  {
    "transaction_id": "TX10002",
    "card_id": "CARD_004",
    "amount": 39.39,
    "merchant": "Amazon",
    "location": "New York",
    "timestamp": "2026-09-28T09:37:27.373071",
    "status": "SUCCESS",
    "scenario": "NORMAL_SPEND"
  }
]


In [2]:
# Demonstrating Fraud Scenarios
print('--- Scenario 1: Rapid Velocity Burst ---')
burst = gen.generate_scenario_rapid_burst('CARD_007', count=6)
print(f'Generated {len(burst)} rapid burst transactions')

print('--- Scenario 2: Impossible Travel ---')
travel = gen.generate_scenario_impossible_travel('CARD_012')
for t in travel:
    print(f'Txn {t["transaction_id"]} at {t["timestamp"]} in {t["location"]}')

print('--- Scenario 3: High Value Abnormal Spike ---')
spike = gen.generate_scenario_high_value_spike('CARD_005')
print(f'Spike Txn: {spike["transaction_id"]} | Amount: ${spike["amount"]}')

print('--- Scenario 4: Account Takeover ---')
takeover = gen.generate_scenario_account_takeover('CARD_018')
print(f'Takeover Txn: {takeover["transaction_id"]} | Location: {takeover["location"]} | Time: {takeover["timestamp"]}')

--- Scenario 1: Rapid Velocity Burst ---
Generated 6 rapid burst transactions
--- Scenario 2: Impossible Travel ---
Txn TX10012 at 2026-09-28T19:44:01.377990 in New York
Txn TX10013 at 2026-09-28T19:59:01.377995 in Singapore
--- Scenario 3: High Value Abnormal Spike ---
Spike Txn: TX10014 | Amount: $695.11
--- Scenario 4: Account Takeover ---
Takeover Txn: TX10015 | Location: Tokyo | Time: 2026-09-28T03:18:42.378122


In [3]:
# Save sample transactions to synthetic storage
all_sample = normal_txns + burst + travel + [spike, takeover]
with open(SYNTHETIC_DATA_PATH, 'w') as f:
    for t in all_sample:
        f.write(json.dumps(t) + '\n')
print(f'Wrote {len(all_sample)} synthetic sample transactions to {SYNTHETIC_DATA_PATH}')

Wrote 15 synthetic sample transactions to /Users/yaseen/Data_Analysis/BDP Project/data/synthetic/streaming_transactions.json


## Notebook Explanation

### What was done
- Built a synthetic transaction stream generator simulating realistic cardholder spending baselines.
- Implemented concrete fraud attack scenarios:
  1. Rapid velocity burst (>5 txns within 2 minutes)
  2. Impossible travel between geographically distant cities (New York to Singapore)
  3. High-value spending spike (>5x cardholder average)
  4. Account takeover at 3 AM from overseas locations
  5. Card testing sequence (declined attempts followed by high-value success)
- Persisted generated sample stream to `data/synthetic/streaming_transactions.json`.

### Why it was done
The public Kaggle dataset lacks card numbers, merchant names, geographic locations, and real-time temporal pacing. Synthetic streaming data provides the necessary attributes to demonstrate dynamic real-time streaming analytics while keeping offline model benchmarking on the real dataset.

### Input
- Cardholder profile distributions and merchant/location metadata.

### Output
- Synthetic stream serialized to `data/synthetic/streaming_transactions.json`.

### Connection to next stage
The synthetic streaming transactions are consumed by `06_streaming_feature_engineering.ipynb` to construct real-time sliding windows and extract personalized behavioral baselines.

### Implementation decisions
- Clearly segregated real data (used for model benchmarking) from synthetic data (used for streaming).
- Structured records in standard JSON format containing required fields (`transaction_id`, `card_id`, `amount`, `merchant`, `location`, `timestamp`).